# P4 Depth Reconstruction — Google Colab

This notebook runs the repository's final hybrid RGB-D depth reconstruction pipeline in Google Colab.

**Recommended runtime:** Runtime → Change runtime type → **T4 GPU**.

The notebook supports:
- RGB guidance images (`.png`, `.jpg`, `.jpeg`)
- low-resolution depth as `.npy` or an image file
- optional ground-truth depth for metrics
- optional SGNet checkpoints for the deep branch
- downloadable `.npy`, preview `.png`, and comparison outputs

If no SGNet checkpoint is provided, the application uses the repository's explicit classical fallback and labels it in the output status.

In [ ]:
#@title 1. Install dependencies and clone the final repository
!git clone -q https://github.com/KumarSrinidhi/CV-Hackthon.git /content/CV-Hackthon 2>/dev/null || true
%cd /content/CV-Hackthon
!pip -q install -r requirements.txt gradio pyyaml

import os
import sys
import torch

print("Repository:", os.getcwd())
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Optional SGNet checkpoint

The checkpoint files are intentionally not stored in Git. To enable the deep/hybrid branch, place the official checkpoint files in `/content/CV-Hackthon/checkpoints/sgnet/model/` using the upload cell below, or mount Google Drive and set `CHECKPOINT_DIR` to the folder containing the `.pth` files.

Expected scale-specific files are the SGNet x4, x8, and x16 checkpoints. If the folder is empty, the notebook still runs the tested classical fallback.

In [ ]:
#@title 2a. Optional: upload SGNet checkpoint files
from google.colab import files
import os
import shutil

CHECKPOINT_DIR = "/content/CV-Hackthon/checkpoints/sgnet/model"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

UPLOAD_CHECKPOINTS = False  #@param {type:"boolean"}
if UPLOAD_CHECKPOINTS:
    uploaded = files.upload()
    for name in uploaded:
        shutil.move(name, os.path.join(CHECKPOINT_DIR, os.path.basename(name)))
    print("Checkpoint files:", sorted(os.listdir(CHECKPOINT_DIR)))
else:
    print("Skipping checkpoint upload.")
os.environ["SGNET_CKPT_DIR"] = "/content/CV-Hackthon/checkpoints/sgnet"
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Files found:", sorted(os.listdir(CHECKPOINT_DIR)))

## 3. Upload inputs

Upload an RGB image and a low-resolution depth map. The RGB dimensions must be exactly `scale` times the depth dimensions, unless the depth has the same dimensions as the RGB image; in that case the app automatically downsamples it.

- Depth `.npy` values should normally be in metres.
- For a 16-bit depth image stored in millimetres, set `DEPTH_SCALE` to `0.001` in the next cell.
- An optional ground-truth depth file enables RMSE, MAE, PSNR, SSIM, boundary RMSE, and edge F1.

In [ ]:
#@title 3a. Upload RGB/depth/ground-truth files
from google.colab import files
import os

uploaded = files.upload()
INPUT_DIR = "/content/CV-Hackthon/colab_inputs"
os.makedirs(INPUT_DIR, exist_ok=True)
for name, data in uploaded.items():
    path = os.path.join(INPUT_DIR, os.path.basename(name))
    with open(path, "wb") as handle:
        handle.write(data)

print("Uploaded files:")
for name in sorted(os.listdir(INPUT_DIR)):
    print(" -", name)

RGB_FILE = ""  #@param {type:"string"}
DEPTH_FILE = ""  #@param {type:"string"}
GT_FILE = ""  #@param {type:"string"}

def choose_file(value, keywords, required=True):
    if value.strip():
        path = os.path.join(INPUT_DIR, value.strip())
        if not os.path.isfile(path):
            raise FileNotFoundError(path)
        return path
    candidates = [
        os.path.join(INPUT_DIR, name)
        for name in sorted(os.listdir(INPUT_DIR))
        if any(word in name.lower() for word in keywords)
    ]
    if required and not candidates:
        raise ValueError(f"No file found for {keywords}. Set the filename parameter explicitly.")
    return candidates[0] if candidates else None

rgb_path = choose_file(RGB_FILE, ("rgb", "color", "image"))
depth_path = choose_file(DEPTH_FILE, ("depth", "lr", "sparse"))
gt_path = choose_file(GT_FILE, ("gt", "ground", "truth"), required=False)
print("RGB:", rgb_path)
print("Depth:", depth_path)
print("Ground truth:", gt_path or "not supplied")

In [ ]:
#@title 4. Run final hybrid reconstruction
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from app.config import load_config
from app.visualization import depth_colormap, error_map, export_npy, export_png
from main import comparison_image, load_depth, load_image, reconstruct

SCALE = 4  #@param [2, 4, 8, 16]
DEPTH_SCALE = 1.0  #@param {type:"number"}
OUTPUT_DIR = "/content/CV-Hackthon/colab_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

rgb = load_image(rgb_path)
lr_depth = load_depth(depth_path, DEPTH_SCALE)
gt = load_depth(gt_path, DEPTH_SCALE) if gt_path else None
config = load_config()

result = reconstruct(rgb, lr_depth, valid_mask=None, scale=int(SCALE), config=config)
prediction = result["depth"]
base = os.path.join(OUTPUT_DIR, f"{result['method']}_x{int(SCALE)}")
npy_path = export_npy(prediction, base + ".npy")
png_path = export_png(prediction, base + ".png")
color_path = base + "_colormap.png"
cv2.imwrite(color_path, depth_colormap(prediction))

comparison = comparison_image(rgb, result, gt, lr_depth)
comparison_path = base + "_comparison.png"
cv2.imwrite(comparison_path, cv2.cvtColor(comparison, cv2.COLOR_RGB2BGR))

metrics = None
error_path = None
if gt is not None:
    if gt.shape != prediction.shape:
        raise ValueError(f"Ground truth shape {gt.shape} does not match prediction {prediction.shape}.")
    from eval.metrics import compute_all_metrics
    metrics = compute_all_metrics(prediction, gt, data_range=config["app"]["data_range"])
    error_path = base + "_error.png"
    cv2.imwrite(error_path, error_map(prediction, gt))

print("method:", result["method"])
print("fallback:", result["fallback"])
for note in result["notes"]:
    print("note:", note)
print("prediction shape:", prediction.shape)
print("prediction range:", float(np.nanmin(prediction)), "to", float(np.nanmax(prediction)))
if metrics:
    print("metrics:")
    for key, value in metrics.items():
        print(f"  {key}: {value:.6f}")

figure, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(rgb)
axes[0].set_title("RGB guidance")
axes[1].imshow(cv2.cvtColor(depth_colormap(prediction), cv2.COLOR_BGR2RGB))
axes[1].set_title(f"{result['method']} depth x{SCALE}")
axes[2].imshow(comparison)
axes[2].set_title("Comparison")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print("Saved:")
for path in (npy_path, png_path, color_path, comparison_path, error_path):
    if path:
        print(" -", path)

In [ ]:
#@title 5. Download the generated files
from google.colab import files
import os

for name in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, name)
    if os.path.isfile(path):
        print("Downloading", name)
        files.download(path)

In [ ]:
#@title 6. Optional: launch the Gradio web app
LAUNCH_GRADIO = False  #@param {type:"boolean"}
if LAUNCH_GRADIO:
    from app.ui import build_app
    demo = build_app()
    demo.launch(share=True, debug=True)

## Notes

- The notebook calls the same `reconstruct()` function as the CLI and Gradio deployment.
- The frozen configuration is loaded from `configs/default_config.yaml`.
- Deep SGNet inference is available only for x4, x8, and x16 and requires the corresponding official checkpoint.
- For a shareable web interface instead of notebook cells, run `!python app.py` in a final cell; the repository's Gradio app will start on the Colab runtime.